# 🧩 Cuaderno 04 — Casos (Unidades 3 y 4)

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Tema 4.4. Casos que combinan **sistemas lineales** (Unidad 3) con **ajuste de curvas** (Unidad 4), y la serie de Fourier como tarea extra.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
# ---------------------------------------------------------------
# En tu computadora usa los archivos locales, sin repositorio.
# En Google Colab pide UNA vez el enlace de tu fork, lo guarda en tu
# Drive y a partir de ahí lo reutiliza solo.
# ---------------------------------------------------------------
import json
import os
import subprocess
import sys

CUADERNO = 'u4_04_casos'
TAREA_ID = 'U4_04_Casos'
REPO_NOMBRE = "U4_Repo"
GUARDADO = os.path.join("CursoMN", "u4_repo.json")   # relativo a MyDrive

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _raiz_local():
    """Sube desde la carpeta actual hasta encontrar lib/robomat.py."""
    candidatos = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatos.append(os.path.dirname(os.path.abspath(nb)))
    candidatos.append(os.getcwd())
    for base in candidatos:
        b = base
        for _ in range(6):
            if os.path.exists(os.path.join(b, "lib", "robomat.py")):
                return b
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


def _normaliza(enlace):
    enlace = (enlace or "").strip().rstrip("/")
    if enlace.endswith(".git"):
        enlace = enlace[:-4]
    if not enlace.startswith("https://github.com/"):
        raise ValueError("Necesito un enlace https://github.com/USUARIO/"
                         + REPO_NOMBRE + " (tu fork).")
    return enlace

# Repositorio del curso (upstream) y carpetas que el alumno nunca edita:
# se refrescan en cada ejecución para que el material y el motor nunca estén
# atrasados, aunque el fork se haya quedado viejo.
CURSO = "https://github.com/ITH-MGL-MN/" + REPO_NOMBRE
MATERIAL = ("lib", "grader", "config", "docs", "ejercicios", "tools")


def _git(*args):
    """Corre git dentro del clon. Devuelve (funcionó, salida)."""
    r = subprocess.run(["git", "-C", REPO] + list(args),
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()


def actualizar():
    """Trae tus commits de GitHub y refresca el material del curso.

    El clon de Drive es desechable: tus respuestas viven en el cuaderno.
    Primero se actualiza el fork (lo que editaste en GitHub) y después se
    copia el material del curso desde `upstream/main`. Si tienes cambios
    locales en el clon, no se tocan: se avisa y se sigue.
    """
    ok, _ = _git("rev-parse", "--is-inside-work-tree")
    if not ok:
        print("⚠️ La carpeta no es un repositorio git: uso los archivos tal cual.")
        return
    _git("remote", "add", "upstream", CURSO + ".git")
    ok, salida = _git("fetch", "--all", "--prune", "--quiet")
    if not ok:
        print("⚠️ No pude conectar con GitHub; sigo con lo que hay en Drive.")
        return

    _git("checkout", "--", *MATERIAL)          # limpia el refresco anterior
    _, rama = _git("rev-parse", "--abbrev-ref", "HEAD")
    rama = rama or "main"
    ok, salida = _git("merge", "--ff-only", "origin/" + rama)
    if ok:
        print("Fork        : al día con tus commits de GitHub")
    else:
        print("⚠️ Tu fork no avanza en línea recta (tienes commits propios")
        print("   o cambios locales): lo dejo como está y solo refresco el")
        print("   material del curso.")

    ok, salida = _git("checkout", "upstream/main", "--", *MATERIAL)
    if ok:
        _git("reset", "--quiet")
        _, sha = _git("rev-parse", "--short", "upstream/main")
        print("Material    : actualizado al curso", sha)
    else:
        print("⚠️ No pude refrescar el material del curso:", salida.splitlines()[:1])


if not ES_COLAB:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError(
            "No encontré la raíz del repositorio (lib/robomat.py).\n"
            "Abre el cuaderno dentro de U4_Repo o ejecútalo desde esa carpeta.")
    print("Modo        : local (sin repositorio)")
    print("Repositorio :", REPO)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    ruta_json = os.path.join(DRIVE, GUARDADO)

    guardado = {}
    if os.path.exists(ruta_json):
        try:
            with open(ruta_json, encoding="utf-8") as f:
                guardado = json.load(f)
        except (OSError, json.JSONDecodeError) as exc:
            raise RuntimeError(
                f"No se pudo leer la configuracion de Drive: {ruta_json}"
            ) from exc

    enlace = (guardado.get("repo") or "").strip()
    if enlace:
        print("✅ Repositorio guardado:", enlace)
        print("   (si cambió, edita la caja de abajo y pulsa el botón)")
    else:
        try:
            enlace = input("Pega el enlace de TU fork de U4_Repo: ")
        except Exception:
            enlace = ""
        enlace = (enlace or "").strip()
        if not enlace:
            raise RuntimeError(
                "Sin repositorio no puedo continuar en Colab.\n"
                "Haz fork del repositorio del curso y pega su enlace.")

    enlace = _normaliza(enlace)
    REPO = os.path.join(DRIVE, "CursoMN", REPO_NOMBRE)

    if not os.path.exists(REPO):
        print("Clonando tu repositorio…")
        subprocess.run(["git", "clone", enlace + ".git", REPO], check=True)
    actualizar()

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump({"repo": enlace}, f, indent=1)
    print("Repositorio :", REPO)
    print("Actualizar  : vuelve a ejecutar esta celda cuando subas cambios a GitHub")

# Instala dependencias del repositorio solo si falta alguna.
import importlib.util

_modulos_requeridos = ("numpy", "scipy", "sympy", "matplotlib", "yaml", "requests")
_faltantes = [nombre for nombre in _modulos_requeridos
              if importlib.util.find_spec(nombre) is None]
if _faltantes:
    requisitos = os.path.join(REPO, "requirements.txt")
    if not os.path.isfile(requisitos):
        raise FileNotFoundError("Falta requirements.txt en el repositorio.")
    print("Instalando dependencias faltantes:", ", ".join(_faltantes))
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", requisitos],
                   check=True)

# Indica al motor empaquetado dónde está la configuración del repositorio.
os.environ["MN_REPO"] = REPO

# --- Motor de calificación -------------------------------------------
# Se usa el código fuente si está (rama `test`, para depurar); si no, el blob
# empaquetado de grader/, que es lo que recibe el alumno.
_ruta_modulos = os.path.join(REPO, "lib", "mn_motor")
_ruta_blob = os.path.join(REPO, "grader", CUADERNO + ".txt")

_lib = os.path.join(REPO, "lib")
if _lib not in sys.path:
    sys.path.insert(0, _lib)          # ajuste_curvas y, si está, mn_motor

if os.path.isfile(os.path.join(_ruta_modulos, "__init__.py")):
    import mn_motor as _motor
    globals().update({n: getattr(_motor, n) for n in _motor.__all__})
    print("Motor       : módulos de lib/mn_motor (modo profesor)")
else:
    import base64
    import zlib

    if not os.path.exists(_ruta_blob):
        raise FileNotFoundError(
            "No encuentro el motor: falta grader/" + CUADERNO + ".txt\n"
            "Clona el repositorio completo (incluye la carpeta grader/).")
    with open(_ruta_blob, encoding="utf-8") as f:
        codigo = zlib.decompress(base64.b64decode(f.read().strip())).decode("utf-8")
    _MOTOR = {"__name__": "mn_motor_blob", "__file__": "<blob>"}
    exec(compile(codigo, "<blob>", "exec"), _MOTOR)
    globals().update({n: _MOTOR[n] for n in _MOTOR["__all__"]})
    print("Motor       : blob verificado de grader/")

usar_cuaderno(CUADERNO, TAREA_ID)

# --- Preludio estilo MATLAB y herramientas de gráficas ---------------
with open(os.path.join(REPO, "lib", "robomat.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

if REPO not in sys.path:
    sys.path.insert(0, REPO)          # para importar ejercicios/
sys.path.append(os.path.join(REPO, "tools"))
import importlib
import visualizar
importlib.reload(visualizar)

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("Preludio    : lib/robomat.py · tools/visualizar.py")
print("\nListo. Ejecuta la celda de identificación.")

In [ ]:
#@title 👤 Identificación (número de control)
# ---------------------------------------------------------------
# En Colab se toma tu correo institucional; en local se escribe uno
# de prueba. De ahí sale el número de control que siembra tus datos.
# ---------------------------------------------------------------
if ES_COLAB:
    import requests

    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    auth.authenticate_user()
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    info = requests.get(
        "https://www.googleapis.com/drive/v3/about?fields=user",
        headers={"Authorization": "Bearer " + creds.token}, timeout=20).json()
    cuenta = info.get("user", {})
    email = cuenta.get("emailAddress", "")
    nombre = cuenta.get("displayName", "")
else:
    email = "m16330887@hermosillo.tecnm.mx"   # <-- cambia por el tuyo
    nombre = ""

if not (email and "@" in email and email.lower().endswith(".tecnm.mx")):
    from IPython.display import HTML, display
    display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;'
        'border-radius:8px;font-weight:bold">⛔ Usa tu correo @….tecnm.mx</p>'))
    raise RuntimeError("Necesito tu correo institucional para sembrar tus datos.")

print("Alumno :", email)
print("NC     :", extraer_nc(email))
MI_TAREA = generar_tarea(email, nombre)
print("Cuaderno:", MI_TAREA["cuaderno"],
      "·", len(MI_TAREA["preguntas"]), "preguntas")

---

## Caso · Calibración de un sensor (mínimos cuadrados + sistemas lineales)

**El problema.** Un sensor entrega una lectura digital $x$ y se calibra contra
un patrón de temperatura $y$. El fabricante reporta ruido de $\pm0.1$ °C.

| $x$ (cuentas) | 0 | 10 | 20 | 30 | 40 | 50 | 60 | 70 | 80 | 90 |
|---|---|---|---|---|---|---|---|---|---|---|
| $y$ (°C) | 0.2 | 0.9 | 1.8 | 2.5 | 3.4 | 4.2 | 5.4 | 6.2 | 7.5 | 8.4 |

**¿Interpolar o ajustar?** Ajustar: hay 10 mediciones con ruido. Interpolar con
grado 9 copiaría el ruido.

**El método más efectivo aquí** es mínimos cuadrados **cuadráticos** (17.19),
porque el lineal deja estructura en los residuales. Y hay un detalle que
conecta las dos unidades: las ecuaciones normales **son un sistema lineal**, así
que se resuelven con lo de la Unidad 3.

**Otros métodos viables.** Lineal (más simple, deja curvatura); cúbico (no
mejora y no se justifica); interpolación (sobreajuste).

In [ ]:
#@title 🧩 Caso A · ecuaciones normales resueltas con Gauss (Unidad 3)
from ajuste_curvas import minimos_cuadrados

x_s = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90], dtype=float)
y_s = np.array([0.2, 0.9, 1.8, 2.5, 3.4, 4.2, 5.4, 6.2, 7.5, 8.4])


def gauss_solve(A, b):
    """Eliminación gaussiana con sustitución hacia atrás (Unidad 3)."""
    A = np.array(A, dtype=float).copy()
    b = np.array(b, dtype=float).copy()
    n = len(b)
    for k in range(n - 1):
        if A[k, k] == 0:
            raise ZeroDivisionError(f"pivote cero en la columna {k + 1}")
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k:] -= m * A[k, k:]
            b[i] -= m * b[k]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
    return x


n_s = len(x_s)
A_norm = np.array([[n_s, x_s.sum(), (x_s ** 2).sum()],
                   [x_s.sum(), (x_s ** 2).sum(), (x_s ** 3).sum()],
                   [(x_s ** 2).sum(), (x_s ** 3).sum(), (x_s ** 4).sum()]])
b_norm = np.array([y_s.sum(), (x_s * y_s).sum(), (x_s ** 2 * y_s).sum()])

coef_gauss = gauss_solve(A_norm, b_norm)
coef_lib = minimos_cuadrados(x_s, y_s, 2)["coef"]
print("con Gauss (U3) :", np.round(coef_gauss, 6))
print("con la librería:", np.round(coef_lib, 6))
print("¿coinciden?     :", np.allclose(coef_gauss, coef_lib, atol=1e-9))

In [ ]:
#@title 🧩 Caso A · ¿lineal o cuadrático? la evidencia
for m in (1, 2):
    r = minimos_cuadrados(x_s, y_s, m)
    print(f"grado {m}: sy/x = {r['syx']:.4f} °C · r2 = {r['r2']:.5f}")

print("\nruido del sensor = 0.1 °C")
print("El lineal queda por encima del ruido y deja curvatura;")
print("el cuadrático baja al orden del ruido: se justifica el término x^2.")

---

## Método 1 · Sistema lineal por eliminación gaussiana (Unidad 3)

Antes de ajustar hay que **resolver**. Las ecuaciones normales de un
ajuste son un sistema lineal, así que se atacan con lo de la Unidad 3:
eliminación hacia adelante y sustitución hacia atrás.

$$a_{ii}^{(k+1)} = a_{ii} - m\,a_{ki}, \qquad m=\frac{a_{ik}}{a_{kk}} \tag{U3}$$

Multiplicadores, pivoteo y costo $O(n^3)$: lo ya estudiado. Aquí sólo se
reutiliza para el caso.

In [ ]:
#@title ✏️ Actividad a mano · Sistema lineal por eliminación gaussiana (Unidad 3)
EJ_1 = mano_enunciado("CASO_GAUSS")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Sistema lineal por eliminación gaussiana (Unidad 3)
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("CASO_GAUSS", [0.0, 0.0, 0.0])

### 💻 Programa la función

Abre `ejercicios/casos/gauss_solve.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Sistema lineal por eliminación gaussiana (Unidad 3)
from ejercicios.casos.gauss_solve import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("CASO_GAUSS")

In [ ]:
#@title 📊 Laboratorio · Sistema lineal por eliminación gaussiana (Unidad 3)
visualizar.laboratorio("CASO_GAUSS", EJ_1)

In [ ]:
#@title 🧪 Pregunta 1 · Sistema lineal por eliminación gaussiana (Unidad 3)
# Cuando tu función ya funcione, contéstala aquí.
pregunta(1)

---

## Método 2 · Calibración: ecuaciones normales por Gauss

El caso completo, con las dos unidades juntas:

1. Un sensor da lecturas $x$ y temperaturas $y$ con ruido.
2. Se plantean las **ecuaciones normales** (17.4)–(17.5): un sistema $2\times2$.
3. Se resuelve con **eliminación gaussiana** (Unidad 3).

$$n a_0+\left(\sum x_i\right)a_1=\sum y_i,\qquad
\left(\sum x_i\right)a_0+\left(\sum x_i^2\right)a_1=\sum x_iy_i \tag{17.4–17.5}$$

Tu función `coeficientes_calibracion` debe **llamar** a `gauss_solve`: no se
resuelve el sistema dos veces con dos métodos distintos.

In [ ]:
#@title ✏️ Actividad a mano · Calibración: ecuaciones normales por Gauss
EJ_2 = mano_enunciado("CASO_NORMALES")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Calibración: ecuaciones normales por Gauss
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("CASO_NORMALES", [0.0, 0.0])

### 💻 Programa la función

Abre `ejercicios/casos/coeficientes_calibracion.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Calibración: ecuaciones normales por Gauss
from ejercicios.casos.coeficientes_calibracion import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("CASO_NORMALES")

In [ ]:
#@title 📊 Laboratorio · Calibración: ecuaciones normales por Gauss
visualizar.laboratorio("CASO_NORMALES", EJ_2)

In [ ]:
#@title 🧪 Pregunta 2 · Calibración: ecuaciones normales por Gauss
# Cuando tu función ya funcione, contéstala aquí.
pregunta(2)

---

## Método 3 · Serie de Fourier (tarea extra)

Con datos igualmente espaciados que completan un periodo, los
coeficientes salen de sumas simples:

$$A_0=\frac{\sum y}{N},\qquad
A_j=\frac{2}{N}\sum y\cos(j\omega_0t),\qquad
B_j=\frac{2}{N}\sum y\operatorname{sen}(j\omega_0t) \tag{19.14–19.16}$$

El modelo es lineal en $A_j$ y $B_j$, así que es mínimos cuadrados con otra
base de funciones.

In [ ]:
#@title ✏️ Actividad a mano · Serie de Fourier (tarea extra)
EJ_3 = mano_enunciado("FOURIER")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Serie de Fourier (tarea extra)
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("FOURIER", [0.0, 0.0, 0.0])

### 💻 Programa la función

Abre `ejercicios/casos/coeficientes_fourier.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Serie de Fourier (tarea extra)
from ejercicios.casos.coeficientes_fourier import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("FOURIER")

In [ ]:
#@title 📊 Laboratorio · Serie de Fourier (tarea extra)
visualizar.laboratorio("FOURIER", EJ_3)

In [ ]:
#@title 🧪 Pregunta 3 · Serie de Fourier (tarea extra)
# Cuando tu función ya funcione, contéstala aquí.
pregunta(3)

In [ ]:
#@title ✅ 1 · Verificación de los ejercicios a mano
mano_comprueba_todos(["CASO_GAUSS", "CASO_NORMALES", "FOURIER"])

In [ ]:
#@title 📝 2 · Verificación de todas las preguntas (no envía nada)
puntos, maximo = calificar()

In [ ]:
#@title 📤 3 · Envío de las respuestas
enviar(email)

In [ ]:
#@title 🔎 4 · Verificación de lo que ya está enviado (no envía nada)
consultar_calificacion()